In [0]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName('Medallion_Architecture').getOrCreate()

Layers in Architecture (Medallion):
- Bronze ➝ Raw data
- Silver ➝ Cleaned + enriched data
- Gold ➝ Aggregated insights

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

Build Medallion Architecture (Practical)

🔶 Bronze – Ingest Raw CSV


In [0]:
raw_df = spark.read.option('header', True).csv('/Workspace/Repos/[email id]/[folder name]/orders.csv')

raw_df.write.format('delta').mode('overwrite').saveAsTable('bronze.orders')

df_orders = spark.table('bronze.orders')

df_orders.printSchema()

display(spark.sql('SELECT * FROM bronze.orders'))

🔷 Silver - Clean and Standard

In [0]:
df_silver = df_orders.withColumns({
    'quantity': F.col('quantity').cast('int'),
    'unit_price': F.col('unit_price').cast('int'),
    'discount': F.col('discount').cast('decimal(3, 2)'),
    'order_date': F.to_date(F.col('order_date'), 'yyyy-MM-dd')
})

df_silver.printSchema()

In [0]:
df_silver = df_silver.dropDuplicates(['order_id'])

df_silver.display()

In [0]:
df_silver = df_silver.withColumn(
    'validation_status',
    F.when(F.col('quantity') <= 0, 'Invalid quantity')
    .when(F.col('unit_price') < 0, 'Invalid unit price')
    .when(~F.col('discount').between(0, 1), 'Invalid discount')
    .otherwise('Valid')
)

display(df_silver)

In [0]:
df_silver = df_silver.withColumn(
    'gross_amount',
    F.col('quantity') * F.col('unit_price')
)

cols = df_silver.columns
unit_price_index = cols.index('unit_price')

cols = (
    cols[:unit_price_index + 1]
    + ['gross_amount']
    + [c for c in cols[unit_price_index + 1:] if c != 'gross_amount']
)

df_silver = df_silver.select(*cols)

display(df_silver)

In [0]:

df_silver = df_silver.withColumns({
    'discount_amount': F.col('gross_amount') * F.col('discount'),
    'net_amount': F.col('gross_amount') - F.col('discount_amount')
})

cols = df_silver.columns
discount_index = cols.index('discount')

cols = (
    cols[:discount_index + 1] 
    + ['discount_amount']
    + ['net_amount']
    + [
        c for c in cols[discount_index + 1:] 
        if c not in ['discount_amount', 'net_amount']
       ]
)

df_silver = df_silver.select(*cols)

display(df_silver)

In [0]:
df_silver = df_silver.withColumn(
    'location',
    F.initcap(F.trim(F.col('location')))
)

display(df_silver)

In [0]:
df_silver = df_silver.withColumn(
    'is_completed',
    F.when(F.col('status') == 'Completed', True)
    .otherwise(False)
)

display(df_silver)

In [0]:
df_silver = df_silver.withColumn(
    'order_size',
    F.when(
        F.col('net_amount') < 2000,
        'Small'
    )
    .when (
        F.col('net_amount').between(2000, 10000),
        'Medium'
    )
    .otherwise('Large')
)

display(df_silver)

In [0]:
df_silver.filter(F.col('customer_name').isNull() | F.col('quantity').isNull() | F.col('gross_amount').isNull()).show()

df_silver.filter(F.col('customer_name').isNotNull()).show()

df_silver = df_silver.fillna({
    'customer_name': 'Unknown',
    'quantity': 0,
    'gross_amount': 0
})

display(df_silver)

In [0]:
df_silver.write.format('delta').mode('overwrite').saveAsTable('silver.orders')

display(spark.sql('SELECT * FROM silver.orders ORDER BY order_id'))

🟡 Gold – Aggregated Insights

In [0]:
df_gold_customer_summary = (
    df_silver.groupBy(['customer_id', 'customer_name'])
    .agg(
        F.sum('quantity').alias('total_orders'),
        F.sum('net_amount').alias('total_spent'),
        F.avg('net_amount').alias('avg_order_value')
    )
)

df_gold_customer_summary.write.format('delta').mode('overwrite').saveAsTable('gold.customer_summary')

display(spark.sql('SELECT * FROM gold.customer_summary'))

In [0]:
df_location_summary = (
    df_silver.groupBy('location') 
    .agg(
        F.sum('quantity').alias('total_orders'),
        F.sum('net_amount').alias('total_revenue'),
        F.round(F.avg('net_amount'), 2).alias('avg_order_values')
    )
    .orderBy('location')
)

df_location_summary.write.format('delta').mode('overwrite').saveAsTable('gold.location_summary')

display(spark.sql('SELECT * FROM gold.location_summary'))

In [0]:
df_product_performance = (
    df_silver.groupBy('product') 
    .agg(
        F.sum('quantity').alias('total_quantity'),
        F.sum('net_amount').alias('total_revenue'),
        F.round(F.avg('net_amount'), 2).alias('avg_order_values')
    ) 
    .orderBy('product')
)

df_product_performance.write.format('delta').mode('overwrite').saveAsTable('gold.product_performance')

display(spark.sql('SELECT * FROM gold.product_performance'))

In [0]:
df_category_performance = (
    df_silver.groupBy('category') 
    .agg(
        F.sum('quantity').alias('total_orders'),
        F.sum('net_amount').alias('total_revenue'),
        F.round(F.avg('net_amount'), 2).alias('avg_order_values')
    )
)

df_category_performance.write.format('delta').mode('overwrite').saveAsTable('gold.category_performance')

display(spark.sql('SELECT * FROM gold.category_performance'))

In [0]:
df_yearly_sales = (
    df_silver.groupBy(F.year(F.col('order_date')).alias('year')) 
    .agg(
        F.sum('net_amount').alias('total_revenue'),
        F.round(F.avg('net_amount'), 2).alias('avg_order_sales')
    ) 
    .orderBy('year')
)

df_yearly_sales.write.format('delta').mode('overwrite').saveAsTable('gold.yearly_sales')

display(spark.sql('SELECT * FROM gold.yearly_sales')) 

In [0]:
df_monthly_sales = (
    df_silver.groupBy(F.monthname(F.col('order_date')).alias('month')) 
    .agg(
        F.sum('net_amount').alias('total_revenue'),
        F.round(F.avg('net_amount'), 2).alias('avg_order_sales')
    ) 
    .orderBy('month')
)

df_monthly_sales.write.format('delta').mode('overwrite').saveAsTable('gold.monthly_sales')

display(spark.sql('SELECT * FROM gold.monthly_sales'))

In [0]:
df_payment_analysis = (
    df_silver.groupBy('payment_method') 
    .agg(
        F.count('payment_method').alias('transaction_count'),
        F.sum('net_amount').alias('total_revenue')
    ) 
    .orderBy(F.col('transaction_count').desc())
)
df_payment_analysis.write.format('delta').mode('overwrite').saveAsTable('gold.payment_analysis')

display(spark.sql('SELECT * FROM gold.payment_analysis'))

In [0]:
df_order_status_summary = (
    df_silver.groupBy('status') 
    .agg(
        F.count('status').alias('order_count'),
        F.sum('net_amount').alias('total_revenue')
    ) 
    .orderBy(F.col('order_count').desc())
)
    
df_order_status_summary.write.format('delta').mode('overwrite').saveAsTable('gold.order_status_summary')

display(spark.sql('SELECT * FROM gold.order_status_summary'))

In [0]:
df_top_customers = (
    df_silver.groupBy('customer_id') 
    .agg(
        F.sum('quantity').alias('order_count'),
        F.sum('net_amount').alias('total_spent'),
        F.round(F.avg('net_amount'), 2).alias('avg_spent'),
        F.max('order_date').alias('last_order_date')
    ) 
    .orderBy(F.col('total_spent').desc()) 
    .limit(5) 
)

df_top_customers.write.format('delta').mode('overwrite').saveAsTable('gold.top_customers')

display(spark.sql('SELECT * FROM gold.top_customers'))

In [0]:
df_customer_segementation = (
    df_silver
    .groupBy('customer_id', 'customer_name')
    .agg(
        F.sum('net_amount').alias('total_spent')
    )
    .withColumn(
        'customer_segment',
        F.when(F.col('total_spent') < 5000, 'Low Value')
        .when(F.col('total_spent') <= 20000, 'Medium Value')
        .otherwise('High Value')
    )
    .select(
        'customer_id',
        'customer_name',
        'total_spent',
        'customer_segment'
    )
)

df_customer_segementation.write.format('delta').mode('overwrite').saveAsTable('gold.customer_segmentation')

spark.sql('SELECT * FROM gold.customer_segmentation').display()

In [0]:
df_daily_sales = (
    df_silver
    .groupBy('order_date')
    .agg(
        F.sum('quantity').alias('orders'),

        F.coalesce(
            F.sum('net_amount'),
            F.lit(0)
        ).alias('daily_revenue'),
    
        F.coalesce(     
            F.round(F.avg('net_amount'), 2),
            F.lit(0)
        ).alias('avg_daily_revenue')
    )
)

df_daily_sales.write.format('delta').mode('overwrite').saveAsTable('gold.daily_sales')

spark.sql('SELECT * FROM gold.daily_sales').display()